---
authors:
  - name: Felix Drost
  - name: Lukas Heumos
    roles:
      - writing – review & editing
---


(air-repertoire-specificity)=
# Specificity analysis
```{dropdown} 🧠 Key takeaways

:::{card}
:link: #air-repertoire-specificity-key-takeaway-1
Specificity is determined mainly by the CDR3 sequences and V genes, and can be inferred from databases, sequence similarity or machine learning models.
:::

:::{card}
:link: #air-repertoire-specificity-key-takeaway-2
TCRdist finds more database matches than exact matching, but the sequence definitions and gene names of both sides have to agree.
:::

:::{card}
:link: #air-repertoire-specificity-key-takeaway-3
Many database matches are coincidental, which the paired gene expression reveals, and predictions for unseen epitopes are close to random.
:::

```
``````{dropdown} ⚙️ Environment setup
`````{tab-set}

````{tab-item} Steps
```{include} ../_static/default_text_env_setup.md
```
````

````{tab-item} yml
```{literalinclude} air_repertoire.yml
:language: yaml
```
````

`````
``````
````{dropdown} 🗄️ Get data and notebooks
```{include} ../_static/default_text_lamindb_setup.md
```
````
<!-- END DROPDOWNS -->


(air-repertoire-specificity-key-takeaway-1)=
## Motivation

The antigen specificity of a receptor is determined mainly by its CDR3 sequences and additionally by its V genes, which encode CDR1 and CDR2 {cite:p}`air:dash2017quantifiable,air:springer2021contribution`.
Specificity can be measured with barcoded peptide-MHC multimers or antigens, but most datasets lack such measurements.
It can instead be inferred by matching receptors to databases of receptors with known specificity, by grouping receptors with similar sequences, or with machine learning models.

In [1]:
import warnings

import anndata as ad
import lamindb as ln
import pandas as pd
import scirpy as ir

warnings.filterwarnings("ignore", category=ad.ExperimentalFeatureWarning)

ln.connect("theislab/sc-best-practices")

ln.track("as8W2VK7byRf")

→ connected lamindb: theislab/sc-best-practices


! It looks like you are running jupyter lab but don't have jupyter-server module installed. Please install it via pip install jupyter-server


→ loaded Transform('as8W2VK7byRf0000', key='specificity.ipynb'), started new Run('a5Azn9RySEgETINU') at 2026-09-29 18:11:18 UTC


We load the T and B cells with the clonotypes from the [previous chapter](clonotype.ipynb).

In [2]:
mdata_tcr = ln.Artifact.get(
    key="air_repertoire/stephenson2021_tcr_clonotypes.h5mu"
).load()
mdata_bcr = ln.Artifact.get(
    key="air_repertoire/stephenson2021_bcr_clonotypes.h5mu"
).load()

(air-repertoire-specificity-key-takeaway-2)=
## Querying VDJdb

VDJdb collects TCRs with experimentally determined epitopes {cite:p}`air:shugay2018vdjdb,air:bagaev2020vdjdb`.
We use the human TCRs of a pinned release, which scirpy obtains from IggyTop with all CDR3 sequences harmonized to junctions, and first search for cells whose α and β chains are identical to an entry.

In [3]:
vdjdb = ln.Artifact.get(key="air_repertoire/vdjdb_human.h5ad").load()
vdjdb

AnnData object with n_obs × n_vars = 175339 × 0
    obs: 'MHC_class', 'MHC_gene_1', 'MHC_gene_2', 'PMID', 'antigen_name', 'antigen_species', 'assay_category', 'assay_method_raw', 'epitope_sequence', 'iedb_iri', 'source_organism', 'tissue', 'source'
    uns: 'DB', 'chain_indices', 'iggytop_metadata'
    obsm: 'airr', 'chain_indices'

In [4]:
ir.pp.ir_dist(mdata_tcr, vdjdb, metric="identity", sequence="aa")
ir.tl.ir_query(
    mdata_tcr,
    vdjdb,
    metric="identity",
    sequence="aa",
    receptor_arms="all",
    dual_ir="any",
)
exact = ir.tl.ir_query_annotate(
    mdata_tcr,
    vdjdb,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen_species"],
    inplace=False,
)

  0%|          | 0/411 [00:00<?, ?it/s]

In [5]:
exact["antigen_species"].value_counts().head(8)

antigen_species
ambiguous                                          142
Epstein-Barr virus                                  67
Cytomegalovirus                                     53
Influenza A virus                                   47
Homo sapiens                                        26
M.tuberculosis                                      22
Severe acute respiratory syndrome coronavirus 2     18
Yellow fever virus                                  12
Name: count, dtype: int64

Few cells share both chains with an entry, since even TCRs against the same epitope rarely have identical sequences.
Cells that match entries of different antigens are labeled as ambiguous.

TCRdist compares CDR3 sequences with an amino acid substitution matrix and gap penalties, and TCRs within a small distance likely recognize the same epitope {cite:p}`air:dash2017quantifiable`.
It grouped TCRs by epitope at least as well as other clustering methods and protein language models in independent comparisons {cite:p}`air:hudson2024comparison,air:nagano2025contrastive`.
We accept matches within a TCRdist of 15 in both chains and require the same V genes.

In [6]:
ir.pp.ir_dist(mdata_tcr, vdjdb, metric="tcrdist", sequence="aa", cutoff=15)
ir.tl.ir_query(
    mdata_tcr,
    vdjdb,
    metric="tcrdist",
    sequence="aa",
    receptor_arms="all",
    dual_ir="any",
    same_v_gene=True,
)
ir.tl.ir_query_annotate(
    mdata_tcr,
    vdjdb,
    metric="tcrdist",
    sequence="aa",
    include_ref_cols=["antigen_species", "epitope_sequence", "MHC_class"],
)
mdata_tcr.pull_obs()

/home/lheumos/miniforge3/envs/air-repertoire/lib/python3.13/site-packages/scirpy/ir_dist/metrics.py:1514: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  data_rows[row_index][0] = data_row_matrix[thread_id, 0:row_end_index].copy()


  0%|          | 0/4575 [00:00<?, ?it/s]

In [7]:
mdata_tcr.obs["airr:antigen_species"].value_counts().head(8)

airr:antigen_species
ambiguous                                          495
Cytomegalovirus                                    312
Epstein-Barr virus                                 196
Influenza A virus                                  152
Homo sapiens                                        87
Severe acute respiratory syndrome coronavirus 2     72
M.tuberculosis                                      53
Human immunodeficiency virus 1                      39
Name: count, dtype: int64[pyarrow]

TCRdist annotates almost four times as many cells as exact matching.
Most matches are to cytomegalovirus, Epstein-Barr virus and influenza A virus, against which most adults carry memory T cells, and about a third of the matched cells match entries of several antigens.

(air-repertoire-specificity-key-takeaway-3)=
## How reliable are the matches?

A match only indicates that a similar TCR bound an epitope in another person, and the paired gene expression lets us test whether the matches are plausible.
Epitopes presented by MHC class I are recognized by CD8+ T cells and those presented by MHC class II by CD4+ T cells.

In [8]:
pd.crosstab(mdata_tcr.obs["airr:MHC_class"], mdata_tcr.obs["gex:cell_type_coarse"])

gex:cell_type_coarse,CD4,CD8,Lymph_prolif,MAIT,Treg,gdT
airr:MHC_class,,,,,,
I,728,437,6,152,65,9
II,50,14,0,0,4,0
ambiguous,42,11,1,4,2,0


Of the cells matched to MHC class I epitopes, 728 are CD4+ T cells, which cannot recognize them, and 437 are CD8+ T cells.
Since CD4+ T cells outnumber CD8+ T cells about 2.5 to 1, the matches are barely enriched in CD8+ T cells, so many of them must be coincidental.

In [9]:
sars_cov_2 = "Severe acute respiratory syndrome coronavirus 2"
donors = (
    mdata_tcr.obs.assign(
        sars_cov_2=lambda obs: obs["airr:antigen_species"].eq(sars_cov_2).fillna(False),
        cytomegalovirus=lambda obs: (
            obs["airr:antigen_species"].eq("Cytomegalovirus").fillna(False)
        ),
    )
    .groupby(["gex:severity", "gex:patient_id"], observed=True)[
        ["sars_cov_2", "cytomegalovirus"]
    ]
    .mean()
)
donors.groupby("gex:severity", observed=True).median()

,sars_cov_2,cytomegalovirus
gex:severity,,
Asymptomatic,0.001297,0.005643
Critical,0.000685,0.008214
Healthy,0.001107,0.005242
Mild,0.001511,0.00418
Moderate,0.000998,0.003855
Severe,0.0,0.006243


If the matches reflected specificity, T cells matching SARS-CoV-2 epitopes should be more frequent in patients than in healthy donors.
Instead, they make up about 0.1% of the T cells of each donor in every group, whereas cytomegalovirus matches are more frequent in all groups.

In [10]:
expansion = pd.crosstab(
    mdata_tcr.obs["airr:antigen_species"],
    mdata_tcr.obs["airr:clonal_expansion"],
    normalize="index",
).loc[["Cytomegalovirus", "Epstein-Barr virus", "Influenza A virus", sars_cov_2]]
expansion.loc["all T cells"] = mdata_tcr.obs["airr:clonal_expansion"].value_counts(
    normalize=True
)
expansion

airr:clonal_expansion,<= 1,<= 2,> 2
airr:antigen_species,,,
Cytomegalovirus,0.794872,0.028846,0.176282
Epstein-Barr virus,0.969388,0.010204,0.020408
Influenza A virus,0.828947,0.000000,0.171053
Severe acute respiratory syndrome coronavirus 2,0.916667,0.055556,0.027778
all T cells,0.875427,0.039123,0.085450


Cells matching cytomegalovirus or influenza A virus epitopes are twice as often part of clonotypes with more than two cells as T cells overall, consistent with the memory T cells that these viruses induce.
Cells matching SARS-CoV-2 epitopes are not expanded, so database matches alone cannot identify the T cells that respond to the infection.

## Predicting specificity

Machine learning models predict whether a TCR binds a given epitope.
In community benchmarks, they performed reasonably for epitopes with many known TCRs but close to random for epitopes that were absent from their training data {cite:p}`air:meysman2023benchmarking,air:nielsen2024lessons`.
Independent evaluations of 21 published predictors and of 50 models confirmed that none generalizes to such unseen epitopes {cite:p}`air:drost2025benchmarking,air:lu2026assessment`.
Their predictions for epitopes outside the training data should therefore be treated as hypotheses that require experimental validation.

## BCR specificity

BCRs bind diverse antigens without MHC restriction, and far fewer BCRs with known specificity are available than TCRs.
CoV-AbDab collects published antibodies against coronaviruses {cite:p}`air:Raybould2021`.
It reports CDR3 sequences without the conserved flanking residues, so we compare them to the `cdr3_aa` field of our BCRs rather than to their junctions, and search for heavy chains with the same CDR3 and V gene.

In [11]:
cov_abdab = ln.Artifact.get(key="air_repertoire/cov_abdab_080224.csv").load()
cov_abdab["v_call"] = cov_abdab["Heavy V Gene"].str.split(" ").str[0]
heavy = ir.get.airr(mdata_bcr, ["cdr3_aa", "v_call"], "VDJ_1")
heavy["VDJ_1_v_call"] = heavy["VDJ_1_v_call"].str.split("*").str[0]
heavy.reset_index().merge(
    cov_abdab, left_on=["VDJ_1_cdr3_aa", "VDJ_1_v_call"], right_on=["CDRH3", "v_call"]
)[["cell_id", "VDJ_1_cdr3_aa", "VDJ_1_v_call", "Name", "Binds to"]]

,cell_id,VDJ_1_cdr3_aa,VDJ_1_v_call,Name,Binds to
0,BGCV13_ATGAGGGGTGCGCTTG-1,ARDLGPYGMDV,IGHV3-66,REGN10957,SARS-CoV2_WT


A single IgG plasma cell of a patient with mild COVID-19 carries the heavy chain CDR3 and V gene of REGN10957, an antibody against SARS-CoV-2, but its light chain differs.
Somatic hypermutation and the diversity of BCRs make such matches rare, so the specificity of B cells is better measured experimentally, for example by sequencing them together with barcoded antigens {cite:p}`air:setliff2019high`.

We store the annotated T cells for the [integration with gene expression](multimodal_integration.ipynb).

In [12]:
ln.Artifact.from_mudata(
    mdata_tcr,
    key="air_repertoire/stephenson2021_tcr_specificity.h5mu",
    description="T cells of the Stephenson 2021 Cambridge samples with VDJdb matches",
).save()

→ returning artifact with same hash: Artifact(uid='3I589652WSqI0YEs0000', key='air_repertoire/stephenson2021_tcr_specificity.h5mu', description='T cells of the Stephenson 2021 Cambridge samples with VDJdb matches', suffix='.h5mu', kind='dataset', otype='MuData', size=814847697, hash='MdD7EGk9EbChbko_zXTqE3', n_files=None, n_observations=54137, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=182, schema_id=None, created_by_id=2, created_at=2026-09-29 18:09:12 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()


Artifact(uid='3I589652WSqI0YEs0000', key='air_repertoire/stephenson2021_tcr_specificity.h5mu', description='T cells of the Stephenson 2021 Cambridge samples with VDJdb matches', suffix='.h5mu', kind='dataset', otype='MuData', size=814847697, hash='MdD7EGk9EbChbko_zXTqE3', n_files=None, n_observations=54137, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=182, schema_id=None, created_by_id=2, created_at=2026-09-29 18:09:12 UTC, is_locked=False, version_tag=None, is_latest=True)

## Quiz

In [13]:
%run ../../scripts/quiz.py

with quiz_tabs():
    flip_card(
        "q1",
        "Why does a match to a database entry not prove the specificity of a T cell?",
        "Different TCRs can match by chance, databases contain noisy entries, and the epitope must also be presented by an MHC molecule that the donor carries.",
        back_font_size=15,
    )
    flip_card(
        "q2",
        "How can gene expression help to check database matches?",
        "Cells matching MHC class I epitopes should be CD8+ T cells and cells matching class II epitopes CD4+ T cells, and T cells against an ongoing infection should be expanded effector cells.",
        back_font_size=15,
    )
    flip_card(
        "q3",
        "Why is it important to know whether a database reports junctions or CDR3 sequences?",
        "Junctions include the conserved flanking residues and CDR3 sequences do not, so sequences only match if both sides use the same definition.",
        back_font_size=15,
    )

## Contributors

We gratefully acknowledge the contributions of:

### Authors

* Felix Drost

### Reviewers

* Lukas Heumos